# OrthoLoC — Colab runner (GPU)

Тонкий раннер: **код живе в GitHub-репо**, тут лише запуск на GPU.

1. Runtime → Change runtime type → GPU.
2. Правки коду — не тут, а в репо (локально / через Claude) → `git push` → у Colab перезапусти клітинку `git pull`.
3. Помилка → скопіюй повний traceback у чат з Claude.

In [ ]:
GITHUB_USER = "somanuTori"   # TODO
BRANCH = "main"
REPO = f"https://github.com/{GITHUB_USER}/OrthoLoC.git"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['ORTHOLOC_CACHE_DIR'] = '/content/drive/MyDrive/ortholoc_cache'   # датасет кешується на Drive між сесіями
OUT = '/content/drive/MyDrive/ortholoc_outputs'
os.makedirs(OUT, exist_ok=True)

In [ ]:
%cd /content
!test -d OrthoLoC || git clone -b $BRANCH $REPO OrthoLoC
%cd /content/OrthoLoC
!git pull --ff-only && git log --oneline -1

In [ ]:
!pip -q install -e ".[gpu]" --extra-index-url https://download.pytorch.org/whl/cu118
import torch; print(torch.__version__, torch.cuda.is_available())

## Санітарний запуск (GT-матчер, без нейромереж)

In [ ]:
!run-localization --sample ortholoc/assets/demo/samples/highway_rural.npz --matcher GT --device cuda --output_dir $OUT/sanity

## Експеримент
Зміни параметри → запусти → рядок у `lab/experiments.md` (команда + коміт нижче).

In [ ]:
MATCHER = "Mast3R"   # див. `run-localization -h` для списку
EXTRA = "--use_adhop"
!git rev-parse --short HEAD
!run-benchmark --dataset_dir ortholoc/assets/demo/samples/ --output_dir $OUT/{MATCHER} --matcher {MATCHER} --device cuda {EXTRA}